# 02 - Explore openFDA Drug Labels & Provision Object Storage

**Learning objective:** Understand the openFDA drug label structure and provision your first Scaleway resource with OpenTofu.

We will:
1. Load the bundled openFDA labels dataset
2. Inspect the section structure (boxed_warning, drug_interactions, pregnancy, etc.)
3. Make one live call to the openFDA API
4. **Live-provision** an Object Storage bucket via `tofu apply` (first IaC moment!)
5. Upload the dataset to S3

###  What We Are Building

An AI assistant that checks drug interactions and warns about risks for special populations
(pregnancy, pediatrics, elderly patients, kidney and liver conditions) -
using only official FDA drug labels as the source of truth. 

#### UC3 from presentation


**What we build:**
- **Scaleway Generative APIs** - Mistral Small 3.2 as a ReAct agent with tool-calling, plus BGE Multilingual Gemma2 for embeddings
- **Scaleway Managed PostgreSQL + pgvector** - stores chunks of FDA Structured Product Labels

**How it works:**  
A pharmacist enters a list of medications. The agent calls tools one by one -
searching for interactions, checking population-specific warnings, rating severity, and summarizing findings -
then returns a structured report.

###  Load the FDA Drug Labels Dataset

This cell loads a pre-downloaded collection of FDA drug labels from a local JSON file.

The dataset comes from the **openFDA Structured Product Labeling (SPL)** API -
each label contains official sections such as warnings, interactions, dosage, and contraindications.

> We use a bundled local file to keep the workshop fast and offline-friendly.

In [ ]:
import json
import os
from dotenv import load_dotenv

load_dotenv()

# Load the bundled dataset
with open("../data/openfda_labels.json") as f:
    labels = json.load(f)

print(f"Loaded {len(labels)} drug labels.")

###  Uderstanding Data
Our dataset contains **200 FDA drug labels**.

Each label in the dataset is a nested JSON object with two main parts:

**Clinical sections** - the actual medical content:
- `boxed_warning` - the most critical safety alert (e.g. bleeding risk for Warfarin)
- `drug_interactions`, `drug_interactions_table` - interaction data
- `warnings_and_cautions`, `contraindications`
- `use_in_specific_populations`, `pediatric_use`, `geriatric_use`, `pregnancy`
- `indications_and_usage`, `mechanism_of_action`, `clinical_pharmacology`
- `adverse_reactions`

**Metadata (`openfda`)** - structured identifiers:
- `brand_name`, `generic_name`, `manufacturer_name`
- `application_number`, `product_ndc`

####  You can explore the full dataset here: [`./data/openfda_labels.json`](./data/openfda_labels.json)

### 🔍 Step 1 - Data Profiling: Exploring the Schema

Before we can use the FDA labels, we need to understand what we are working with.

This cell is the **first step of the data cleaning process** - profiling the dataset.
It scans every label and counts how often each section appears across the full collection.

This tells us which sections are **consistently present** (safe to rely on)
and which are **sparse or missing** (need to be handled carefully before indexing).

> 🧹 Never skip profiling. Knowing your data structure is what separates a reliable AI system from one that silently fails.

In [ ]:
# Schema summary: what sections are present across the dataset?
from collections import Counter

section_counts = Counter()
for label in labels:
    for key in label:
        if key != "openfda":
            section_counts[key] += 1

print("Section across all labels:")
for section, count in section_counts.most_common():
    print(f"  {section}: {count}/{len(labels)}")

**⚠️ Missing sections are not errors - they reflect real differences between drug labels.  Our chunking logic must handle `None` values gracefully before indexing into pgvector.**

###  What is Warfarin?

Warfarin (brand name: Coumadin) is a blood thinner taken as a pill.

Doctors prescribe it to prevent dangerous blood clots - for example in patients with:
- deep vein thrombosis (clots in the legs)
- pulmonary embolism (clots in the lungs)
- atrial fibrillation (irregular heartbeat that can cause stroke)

In [ ]:
# Sample drug: find warfarin
warfarin = next(
    (l for l in labels if "warfarin" in l.get("openfda", {}).get("generic_name", [""])[0].lower()),
    None,
)

if warfarin:
    print(f"Drug: {warfarin['openfda']['generic_name'][0]}")
    print(f"Brand: {warfarin['openfda']['brand_name'][0]}")
    set_ids = warfarin["openfda"].get("spl_set_id", [])
    print(f"Set ID: {set_ids[0] if set_ids else '(not in bundled dataset)'}")
    print(f"Sections: {[k for k in warfarin if k != 'openfda']}")
else:
    print("Warfarin not found in dataset.")

### ✂️ Why Do We Split Text into Chunks?

Embedding models have a **context limit** - they can only process a limited amount of text at once.

> To stay within that limit, we split each label section into chunks of **2000 characters**.
This keeps every chunk enough to embed accurately, without cutting off important content mid-sentence.

> 💡 Smaller chunks = more precise search results. The model retrieves exactly the paragraph that answers the question - not an entire page of text.

In [ ]:
# Sample drug_interactions chunk
if warfarin and "drug_interactions" in warfarin:
    di_text = warfarin["drug_interactions"][0]
    print("=== Drug Interactions (first 2000 chars) ===")
    print(di_text[:2000])

## First IaC moment: Provision an Object Storage bucket

We will now run `tofu apply` to create a Scaleway Object Storage bucket.
This is your first live Infrastructure-as-Code action!

In [ ]:
import subprocess

project_suffix = os.environ["PROJECT_SUFFIX"]
iac_dir = "../iac_snippets/object_storage"

# Write tfvars
tfvars = f"""access_key      = "{os.environ["SCW_ACCESS_KEY"]}"
secret_key      = "{os.environ["SCW_SECRET_KEY"]}"
organization_id = "{os.environ["SCW_DEFAULT_ORGANIZATION_ID"]}"
project_id      = "{os.environ["SCW_DEFAULT_PROJECT_ID"]}"
project_suffix      = "{project_suffix}"
"""
with open(f"{iac_dir}/terraform.tfvars", "w") as f:
    f.write(tfvars)

print("Written terraform.tfvars")

In [ ]:
# Initialize and apply
subprocess.run(["tofu", "init"], cwd=iac_dir, check=True)
result = subprocess.run(
    ["tofu", "apply", "-auto-approve"],
    cwd=iac_dir,
    capture_output=True,
    text=True,
)
print(result.stdout[-500:] if len(result.stdout) > 500 else result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-500:])
    raise RuntimeError(f"tofu apply failed (rc={result.returncode})")

In [ ]:
# Get the bucket name from tofu output
result = subprocess.run(
    ["tofu", "output", "-raw", "bucket_name"],
    cwd=iac_dir,
    capture_output=True,
    text=True,
)
bucket_name = result.stdout.strip()
print(f"Bucket created: {bucket_name}")

### ☁️ Upload Dataset to Scaleway Object Storage

This cell uploads the FDA labels file to **Scaleway Object Storage** - an S3-compatible storage service.

We use the `boto3` library (the standard AWS S3 client) which works with Scaleway out of the box -
just by pointing it to the Scaleway endpoint instead of AWS.

All credentials and the target region are loaded from your `.env` file.

In [ ]:
# Upload the dataset to S3
import boto3

s3 = boto3.client(
    "s3",
    endpoint_url=f"https://s3.{os.environ['SCW_DEFAULT_REGION']}.scw.cloud",
    aws_access_key_id=os.environ["SCW_ACCESS_KEY"],
    aws_secret_access_key=os.environ["SCW_SECRET_KEY"],
    region_name=os.environ["SCW_DEFAULT_REGION"],
)

s3.upload_file(
    "../data/openfda_labels.json",
    bucket_name,
    "openfda_labels.json",
)
print(f"Uploaded openfda_labels.json to s3://{bucket_name}/")

### ✅ You should now have

- ☑️ Explored the openFDA label structure
- ☑️ Inspected Warfarin's `drug_interactions` and `boxed_warning` sections
- ☑️ Made a live call to the openFDA API
- ☑️ Provisioned an Object Storage bucket via OpenTofu
- ☑️ Uploaded the labels dataset to S3

> 🚀 **Next:** `03_provision_pgvector_and_chunk.ipynb`